In [1]:
import os
import sys
from functools import partial
from itertools import product

import numpy as np

import GPy
from sklearn.metrics.pairwise import euclidean_distances

import matplotlib.pyplot as plt
plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'
plt.style.use('seaborn-v0_8-paper')
plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.titlesize": 16,
    "legend.fontsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
})

from BO_loop import BO_loop_RS_diabetes
from BO_loop import BO_loop_RO_diabetes
from adv_RO import find_x_ro_action_diabetes_contextual
from adv_RS1 import find_x_rs_action_RS1_diabetes_contextual
from adv_RS2 import find_x_rs_action_RS2_diabetes_contextual

In [2]:
# Define the ranges
insulin_values = np.linspace(0, 15, 16).reshape(-1,1)
carb_values = np.linspace(20, 100, 20).reshape(-1,1)
fbg_values = np.linspace(90, 130, 5).reshape(-1,1)

cost_matrix_dir  = "util_arrays"

In [ ]:
cost_matrix_dir  = "util_arrays"

# Load the cost matrix
cost_matrix = np.load(f"{cost_matrix_dir}/pbg_obj_func_110.npy")

cost_matrix = cost_matrix[:, :, 1:2]
fbg_values = fbg_values[1:2]

print(cost_matrix.shape)

In [4]:
def ins_index_to_flat_index(ins_ind, cho_ind, fbg_ind, len_cho, len_fbg):
    """
    Convert the 3D index to a flat index.
    """
    return (ins_ind * len_cho * len_fbg) + (cho_ind * len_fbg) + fbg_ind

In [ ]:
distance_metric = partial(euclidean_distances, squared=True)

# Standardize the cost matrix
f_objective = cost_matrix
# Standardize the f_objective using mean and std.
obj_mean = f_objective.mean()
obj_std = f_objective.std()
f_objective = (f_objective - obj_mean) / obj_std

# Print function min, max and avg in one line
print(f"Min: {f_objective.min():.4f}, Max: {f_objective.max():.4f}, Avg: {f_objective.mean():.4f}")

X = np.array(list(product(insulin_values, carb_values, fbg_values)))
X_flat = X.reshape(-1, 3)
Y = f_objective.flatten()
Y_mat = f_objective

tau = (-10.0 - obj_mean) / obj_std
epsilon_t = 10.0
c_ref_ind = 3
c_true_ind = 1
fbg_true_ind = 0
r_ro_true = epsilon_t * 1.0
r_ro_large = epsilon_t * 10.0
r_ro_small = epsilon_t * (1.0/10)

index_ins_ro_true = find_x_ro_action_diabetes_contextual(
    C=carb_values, Y=f_objective[:,:,fbg_true_ind], ind_ref_cho=c_ref_ind, r_ro=r_ro_true**2,
    dist_metric=distance_metric
)
index_x_ro_true = ins_index_to_flat_index(
    index_ins_ro_true, c_ref_ind, fbg_true_ind, len(carb_values), len(fbg_values)
)
index_ins_ro_large = find_x_ro_action_diabetes_contextual(
    C=carb_values, Y=f_objective[:,:,fbg_true_ind], ind_ref_cho=c_ref_ind, r_ro=r_ro_large**2,
    dist_metric=distance_metric
)
index_x_ro_large = ins_index_to_flat_index(
    index_ins_ro_large, c_ref_ind, fbg_true_ind, len(carb_values), len(fbg_values)
)
index_ins_ro_small = find_x_ro_action_diabetes_contextual(
    C=carb_values, Y=f_objective[:,:,fbg_true_ind], ind_ref_cho=c_ref_ind, r_ro=r_ro_small**2,
    dist_metric=distance_metric
)
index_x_ro_small = ins_index_to_flat_index(
    index_ins_ro_small, c_ref_ind, fbg_true_ind, len(carb_values), len(fbg_values)
)
index_ins_rs1, kappa = find_x_rs_action_RS1_diabetes_contextual(
    C=carb_values, Y=f_objective[:,:,fbg_true_ind], ind_ref_cho=c_ref_ind, tau=tau,
    dist_metric=euclidean_distances, pow=1
)
index_x_rs1 = ins_index_to_flat_index(
    index_ins_rs1, c_ref_ind, fbg_true_ind, len(carb_values), len(fbg_values)
)
index_ins_rs2, epsilon_sqr = find_x_rs_action_RS2_diabetes_contextual(
    C=carb_values, Y=f_objective[:,:,fbg_true_ind], ind_ref_cho=c_ref_ind, tau=tau,
    dist_metric=distance_metric
)
index_x_rs2 = ins_index_to_flat_index(
    index_ins_rs2, c_ref_ind, fbg_true_ind, len(carb_values), len(fbg_values)
)

print(f"Tau: {tau:.4f}")
print(f"RS1: {X[index_x_rs1].flatten().tolist()}")
print(f"RS2: {X[index_x_rs2].flatten().tolist()}")
print(f"RO True: {X[index_x_ro_true].flatten().tolist()}")
print(f"RO Large: {X[index_x_ro_large].flatten().tolist()}")
print(f"RO Small: {X[index_x_ro_small].flatten().tolist()}")
# Print epsilon and kappa
print(f"RS2 epsilon: {np.sqrt(epsilon_sqr)}")
print("Percentage of points above tau:", np.mean(Y >= tau))

In [ ]:
kern = GPy.kern.RBF(input_dim=3, variance=1.0, lengthscale=[3.0, 6.0, 1.0], ARD=True)

T = 500
tau = (-10.0 - obj_mean) / obj_std
shift_std = 8
shift_std_in_carb = ((carb_values.max()-carb_values.min()) / len(carb_values)) * shift_std
epsilon_t = shift_std_in_carb * 1.0
r_ro_true = shift_std_in_carb * 1.0
r_ro_large = shift_std_in_carb * 3
r_ro_small = shift_std_in_carb * (1.0/3)
distance_metric = partial(euclidean_distances, squared=True)
sigma_noise = 1
num_initial_points = 5
num_iter = 100

path = "output_arrays"
os.makedirs(path, exist_ok=True)
Y = Y.reshape(-1, 1)
print(Y.shape)

SHOULD_SAVE = False

# Name the arrays to save
if np.isscalar(epsilon_t):
    array_RS1_name = f'diabetes_N_{num_iter}_tau_{tau:.3f}_epsilon_{epsilon_t}_sigma_{sigma_noise:.0e}_T_{T}_RS1.npy'
    array_RS2_name = f'diabetes_N_{num_iter}_tau_{tau:.3f}_epsilon_{epsilon_t}_sigma_{sigma_noise:.0e}_T_{T}_RS2.npy'
    array_RO_true_name = f'diabetes_N_{num_iter}_r_{r_ro_true:.2f}_epsilon_{epsilon_t}_sigma_{sigma_noise:.0e}_T_{T}_RO.npy'
    array_RO_small_name = f'diabetes_N_{num_iter}_r_{r_ro_small:.2f}_epsilon_{epsilon_t}_sigma_{sigma_noise:.0e}_T_{T}_RO.npy'
    array_RO_large_name = f'diabetes_N_{num_iter}_r_{r_ro_large:.2f}_epsilon_{epsilon_t}_sigma_{sigma_noise:.0e}_T_{T}_RO.npy'
elif isinstance(epsilon_t, np.ndarray):
    array_RS1_name = f'diabetes_N_{num_iter}_tau_{tau:.3f}_epsilon_arr_sigma_{sigma_noise:.0e}_T_{T}_RS1.npy'
    array_RS2_name = f'diabetes_N_{num_iter}_tau_{tau:.3f}_epsilon_arr_sigma_{sigma_noise:.0e}_T_{T}_RS2.npy'
    array_RO_true_name = f'diabetes_N_{num_iter}_r_arr_epsilon_arr_sigma_{sigma_noise:.0e}_T_{T}_RO.npy'
    array_RO_small_name = f'diabetes_N_{num_iter}_r_arr_epsilon_arr_sigma_{sigma_noise:.0e}_T_{T}_RO.npy'
    array_RO_large_name = f'diabetes_N_{num_iter}_r_arr_epsilon_arr_sigma_{sigma_noise:.0e}_T_{T}_RO.npy'

array_C_true_inds_name = f'diabetes_N_{num_iter}_T_{T}_C_true_inds.npy'
array_C_ref_inds_name = f'diabetes_N_{num_iter}_T_{T}_C_ref_inds.npy'
array_fbg_true_inds_name = f'diabetes_N_{num_iter}_T_{T}_fbg_true_inds.npy'

sys.exit("Comment out to run the BO loops")

C_true_inds = np.random.choice(len(carb_values), size=(num_iter, T), replace=True)  # num_iter by T np.ndarray
# C_ref_shifts = np.random.randint(low=-12, high=13, size=(num_iter, T))
C_ref_shifts = np.round(np.random.normal(0, shift_std, size=(num_iter, T))).astype(int)
C_ref_inds = np.clip(C_true_inds + C_ref_shifts, a_min=0, a_max=len(carb_values)-1)  # num_iter by T np.ndarray
fbg_true_inds = np.random.choice(len(fbg_values), size=(num_iter, T), replace=True)  # num_iter by T np.ndarray

if SHOULD_SAVE:
    np.save(os.path.join(path, array_C_true_inds_name), C_true_inds)
    np.save(os.path.join(path, array_C_ref_inds_name), C_ref_inds)
    np.save(os.path.join(path, array_fbg_true_inds_name), fbg_true_inds)

# Start the BO loops
print(f'Num_iter: {num_iter}')

print('Starting RS1 loop')
history_RS1 = BO_loop_RS_diabetes(
    X=X_flat, Y=Y_mat, C_ref_inds=C_ref_inds, C_true_inds=C_true_inds, fbg_true_inds=fbg_true_inds,
    acquisition_function=find_x_rs_action_RS1_diabetes_contextual, kernel=kern, sigma_noise=sigma_noise,
    num_init_samples=num_initial_points, distance_metric=euclidean_distances,
    tau=tau, T=T,
    print_samples=False, num_iter=num_iter
)
# Save the history array of RS1
if SHOULD_SAVE:
    np.save(os.path.join(path, array_RS1_name), history_RS1)

print('Starting RS2 loop')
history_RS2 = BO_loop_RS_diabetes(
    X=X_flat, Y=Y_mat, C_ref_inds=C_ref_inds, C_true_inds=C_true_inds, fbg_true_inds=fbg_true_inds,
    acquisition_function=find_x_rs_action_RS2_diabetes_contextual, kernel=kern, sigma_noise=sigma_noise,
    num_init_samples=num_initial_points, distance_metric=distance_metric,
    tau=tau, T=T,
    print_samples=False, num_iter=num_iter
)
# Save the history array of RS2
if SHOULD_SAVE:
    np.save(os.path.join(path, array_RS2_name), history_RS2)

print('Starting RO_true loop')
history_RO_true = BO_loop_RO_diabetes(
    X=X_flat, Y=Y_mat, C_ref_inds=C_ref_inds, C_true_inds=C_true_inds, fbg_true_inds=fbg_true_inds,
    num_init_samples=num_initial_points,
    kernel=kern, distance_metric=distance_metric, sigma_noise=sigma_noise,
    r_ro=r_ro_true**2, T=T,
    print_samples=False, num_iter=num_iter
) 
# Save the history array of RO_true
if SHOULD_SAVE:
    np.save(os.path.join(path, array_RO_true_name), history_RO_true)

print('Starting RO_small loop')
history_RO_small = BO_loop_RO_diabetes(
    X=X_flat, Y=Y_mat, C_ref_inds=C_ref_inds, C_true_inds=C_true_inds, fbg_true_inds=fbg_true_inds,
    num_init_samples=num_initial_points,
    kernel=kern, distance_metric=distance_metric, sigma_noise=sigma_noise,
    r_ro=r_ro_small**2, T=T,
    print_samples=False, num_iter=num_iter
) 
# Save the history array of RO_small
if SHOULD_SAVE:
    np.save(os.path.join(path, array_RO_small_name), history_RO_small)

print('Starting RO_large loop')
history_RO_large = BO_loop_RO_diabetes(
    X=X_flat, Y=Y_mat, C_ref_inds=C_ref_inds, C_true_inds=C_true_inds, fbg_true_inds=fbg_true_inds,
    num_init_samples=num_initial_points,
    kernel=kern, distance_metric=distance_metric, sigma_noise=sigma_noise,
    r_ro=r_ro_large**2, T=T,
    print_samples=False, num_iter=num_iter
) 
# Save the history array of RO_large
if SHOULD_SAVE:
    np.save(os.path.join(path, array_RO_large_name), history_RO_large)


In [7]:
num_iter = 100
T = 500

folder_addition="old_diabetes/"

# Load output arrays for plotting
data_C_true_inds = np.load(f'output_arrays/{folder_addition}diabetes_N_{num_iter}_T_{T}_C_true_inds.npy')
data_C_ref_inds = np.load(f'output_arrays/{folder_addition}diabetes_N_{num_iter}_T_{T}_C_ref_inds.npy')
data_fbg_true_inds = np.load(f'output_arrays/{folder_addition}diabetes_N_{num_iter}_T_{T}_fbg_true_inds.npy')

# Calculate kappa*epsilon values
carb_scale = ((carb_values.max() - carb_values.min()) / len(carb_values))
data_kappa_tau = np.empty((num_iter, T))
data_epsilon_t = np.empty((num_iter, T))
for i in range(num_iter):
    for t in range(T):
        _, kappa_tau = find_x_rs_action_RS1_diabetes_contextual(
            C=carb_values, Y=f_objective[:,:,data_fbg_true_inds[i, t]],
            ind_ref_cho=data_C_ref_inds[i, t], tau=tau, dist_metric=euclidean_distances
        )
        data_kappa_tau[i, t] = kappa_tau
        epsilon_t = np.abs(data_C_true_inds[i, t] - data_C_ref_inds[i, t]) * carb_scale
        data_epsilon_t[i, t] = epsilon_t
data_kappa_epsilon = np.maximum(data_kappa_tau * data_epsilon_t, 0)

data_RS1 = np.load(f'output_arrays/{folder_addition}diabetes_N_{num_iter}_tau_-0.127_epsilon_32.0_sigma_1e+00_T_{T}_RS1.npy', allow_pickle=True)
data_RS2 = np.load(f'output_arrays/{folder_addition}diabetes_N_{num_iter}_tau_-0.127_epsilon_32.0_sigma_1e+00_T_{T}_RS2.npy', allow_pickle=True)
data_RO_true = np.load(f'output_arrays/{folder_addition}diabetes_N_{num_iter}_r_32.00_epsilon_32.0_sigma_1e+00_T_{T}_RO.npy', allow_pickle=True)
data_RO_small = np.load(f'output_arrays/{folder_addition}diabetes_N_{num_iter}_r_10.67_epsilon_32.0_sigma_1e+00_T_{T}_RO.npy', allow_pickle=True)
data_RO_large = np.load(f'output_arrays/{folder_addition}diabetes_N_{num_iter}_r_96.00_epsilon_32.0_sigma_1e+00_T_{T}_RO.npy', allow_pickle=True)

In [8]:
# Lenient regrets
cum_regret_RS1 = np.cumsum(np.maximum(tau - Y[data_RS1[:,1,:].astype(int)], 0), axis=1)
cum_regret_RS2 = np.cumsum(np.maximum(tau - Y[data_RS2[:,1,:].astype(int)], 0), axis=1)
cum_regret_RO_true = np.cumsum(np.maximum(tau - Y[data_RO_true[:,1,:].astype(int)], 0), axis=1)
cum_regret_RO_small = np.cumsum(np.maximum(tau - Y[data_RO_small[:,1,:].astype(int)], 0), axis=1)
cum_regret_RO_large = np.cumsum(np.maximum(tau - Y[data_RO_large[:,1,:].astype(int)], 0), axis=1)

# RS regrets
cum_rs_regret_RS1 = np.cumsum(np.maximum(tau - data_kappa_epsilon[..., None] - Y[data_RS1[:,1,:].astype(int)], 0), axis=1)
cum_rs_regret_RS2 = np.cumsum(np.maximum(tau - data_kappa_epsilon[..., None] - Y[data_RS2[:,1,:].astype(int)], 0), axis=1)
cum_rs_regret_RO_true = np.cumsum(np.maximum(tau - data_kappa_epsilon[..., None] - Y[data_RO_true[:,1,:].astype(int)], 0), axis=1)
cum_rs_regret_RO_small = np.cumsum(np.maximum(tau - data_kappa_epsilon[..., None] - Y[data_RO_small[:,1,:].astype(int)], 0), axis=1)
cum_rs_regret_RO_large = np.cumsum(np.maximum(tau - data_kappa_epsilon[..., None] - Y[data_RO_large[:,1,:].astype(int)], 0), axis=1)

# Calculate average cumulative regrets
avg_cum_reg_list = [
    np.mean(cum_regret_RS1, axis=0),
    np.mean(cum_regret_RS2, axis=0),
    np.mean(cum_regret_RO_true, axis=0),
    np.mean(cum_regret_RO_small, axis=0),
    np.mean(cum_regret_RO_large, axis=0)
]
avg_cum_rs_reg_list = [
    np.mean(cum_rs_regret_RS1, axis=0),
    np.mean(cum_rs_regret_RS2, axis=0),
    np.mean(cum_rs_regret_RO_true, axis=0),
    np.mean(cum_rs_regret_RO_small, axis=0),
    np.mean(cum_rs_regret_RO_large, axis=0)
]

# Calculate variance for error bars
var_cum_reg_list = [
    np.std(cum_regret_RS1, axis=0),
    np.std(cum_regret_RS2, axis=0),
    np.std(cum_regret_RO_true, axis=0),
    np.std(cum_regret_RO_small, axis=0),
    np.std(cum_regret_RO_large, axis=0)
]
var_cum_rs_reg_list = [
    np.std(cum_rs_regret_RS1, axis=0),
    np.std(cum_rs_regret_RS2, axis=0),
    np.std(cum_rs_regret_RO_true, axis=0),
    np.std(cum_rs_regret_RO_small, axis=0),
    np.std(cum_rs_regret_RO_large, axis=0)
]

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(5, 3.5), squeeze=False)
alpha_error = 0.5
markevery = 16 * (T//100)

# Plotting list_0 means
axs[0, 0].plot(avg_cum_reg_list[0], label='RS1', c='r', linestyle='--', marker='^', markevery=markevery, markersize=8)
axs[0, 0].plot(avg_cum_reg_list[1], label='RS2', c='m', linestyle='--', marker='^', markevery=markevery, markersize=8)
axs[0, 0].plot(avg_cum_reg_list[2], label=r'RO $r=\sigma_s$', c='b', linestyle='--', marker='d', markevery=markevery, markersize=8)
axs[0, 0].plot(avg_cum_reg_list[3], label=r'RO $r=\sigma_s~ /~ 3$', c='g', linestyle='--', marker='d', markevery=markevery, markersize=8)
axs[0, 0].plot(avg_cum_reg_list[4], label=r'RO $r=3 \cdot \sigma_s$', c='darkred', linestyle='--', marker='d', markevery=markevery, markersize=8)
# Plotting list error bars at marker points
axs[0, 0].errorbar(range(0, T, markevery), avg_cum_reg_list[0][::markevery].flatten(),
                yerr=var_cum_reg_list[0][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='r', 
)
axs[0, 0].errorbar(range(0, T, markevery), avg_cum_reg_list[1][::markevery].flatten(),
                yerr=var_cum_reg_list[1][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='m', 
)
axs[0, 0].errorbar(range(0, T, markevery), avg_cum_reg_list[2][::markevery].flatten(),
                yerr=var_cum_reg_list[2][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='b'
)
axs[0, 0].errorbar(range(0, T, markevery), avg_cum_reg_list[3][::markevery].flatten(),
                yerr=var_cum_reg_list[3][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='g'
)
axs[0, 0].errorbar(range(0, T, markevery), avg_cum_reg_list[4][::markevery].flatten(),
                yerr=var_cum_reg_list[4][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='darkred'
)
# Adding labels and title
axs[0, 0].set_xlabel('Round t')
axs[0, 0].set_ylabel('Lenient Regret')
axs[0, 0].set_title(' ')

# Adding legend and adjusting layout
axs[0, 0].legend()
# Adding legends for tau and epsilon_t
axs[0, 0].legend(title=r'$\tau=-10$, $\epsilon_t \sim \mathcal{N}(0, \sigma_s=32)$')#, loc='lower right')

plt.tight_layout()
# Save as pdf
dir = 'figures'
os.makedirs(dir, exist_ok=True)
# plt.savefig(os.path.join(dir, f'regret_diabetes_N_{num_iter}.pdf'), format='pdf', bbox_inches='tight')
# Display the plot
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(5, 3.5), squeeze=False)
alpha_error = 0.5
markevery = 16 * (T//100)

# Plotting list_0 means
axs[0, 0].plot(avg_cum_rs_reg_list[0], label='RS1', c='r', linestyle='--', marker='^', markevery=markevery, markersize=8)
axs[0, 0].plot(avg_cum_rs_reg_list[1], label='RS2', c='m', linestyle='--', marker='^', markevery=markevery, markersize=8)
axs[0, 0].plot(avg_cum_rs_reg_list[2], label=r'RO $r=\sigma_s$', c='b', linestyle='--', marker='d', markevery=markevery, markersize=8)
axs[0, 0].plot(avg_cum_rs_reg_list[3], label=r'RO $r=\sigma_s~ /~ 3$', c='g', linestyle='--', marker='d', markevery=markevery, markersize=8)
axs[0, 0].plot(avg_cum_rs_reg_list[4], label=r'RO $r=3 \cdot \sigma_s$', c='darkred', linestyle='--', marker='d', markevery=markevery, markersize=8)
# Plotting list error bars at marker points
axs[0, 0].errorbar(range(0, T, markevery), avg_cum_rs_reg_list[0][::markevery].flatten(),
                yerr=var_cum_rs_reg_list[0][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='r', 
)
axs[0, 0].errorbar(range(0, T, markevery), avg_cum_rs_reg_list[1][::markevery].flatten(),
                yerr=var_cum_rs_reg_list[1][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='m', 
)
axs[0, 0].errorbar(range(0, T, markevery), avg_cum_rs_reg_list[2][::markevery].flatten(),
                yerr=var_cum_rs_reg_list[2][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='b'
)
axs[0, 0].errorbar(range(0, T, markevery), avg_cum_rs_reg_list[3][::markevery].flatten(),
                yerr=var_cum_rs_reg_list[3][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='g'
)
axs[0, 0].errorbar(range(0, T, markevery), avg_cum_rs_reg_list[4][::markevery].flatten(),
                yerr=var_cum_rs_reg_list[4][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='darkred'
)
# Adding labels and title
axs[0, 0].set_xlabel('Round t')
axs[0, 0].set_ylabel('Robust Satisficing Regret')
axs[0, 0].set_title(' ')

# Adding legend and adjusting layout
axs[0, 0].legend()
# Adding legends for tau and epsilon_t
axs[0, 0].legend(title=r'$\tau=-10$, $\epsilon_t \sim \mathcal{N}(0, \sigma_s=32)$')

plt.tight_layout()
# Save as pdf
dir = 'figures'
os.makedirs(dir, exist_ok=True)
# plt.savefig(os.path.join(dir, f'rs_regret_diabetes_N_{num_iter}.pdf'), format='pdf', bbox_inches='tight')
# Display the plot
plt.show()